# NB09 – Consolidate + pair-level split (v2)

Run **once, after all six generators finish.** Reconciles true counts from shards
(expects 10 K real + 6 × 10 K AI = 70 K total), deduplicates, and builds a
deterministic **pair-level** 70/15/15 split where every real image and its six
AI partners land in the same split. Outputs `manifest.parquet` + `splits.parquet`.

GPU not needed.

In [1]:
import importlib.util, sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
_need = []
for _m, _p in [("huggingface_hub","huggingface_hub>=0.23"),
               ("pyarrow","pyarrow"), ("PIL","pillow"), ("datasets","datasets")]:
    if importlib.util.find_spec(_m) is None: _need.append(_p)
if _need: _pip(*_need); print("installed:", _need)
else: print("all base deps present")

all base deps present


In [2]:
REPO_ID = "Shanmuk4622/ai-detection-dataset-v2"
import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
MASTER_SEED = cfg.get("master_seed", 42)   # change master_seed in NB00 to reseed everything
print(f"library {C.PIPELINE_VERSION} loaded | MASTER_SEED={MASTER_SEED}")
print("generators:", list(cfg["generators"]))

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

library 1.2 loaded | MASTER_SEED=42
generators: ['sd15', 'sdxl', 'flux_schnell', 'kandinsky22', 'pixart_sigma', 'sd_cascade']


In [3]:
# gather metadata (no image bytes) from every prefix
import hashlib
from collections import Counter, defaultdict

prefixes = ["real/"] + [f"data/{m}/" for m in cfg["generators"]]
rows = []   # (image_id, source_real_id, generator, label, sha256)
for p in prefixes:
    shards = C.list_shards(REPO_ID, p, TOKEN); cnt = 0
    for f in shards:
        loc = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
        t   = C.pq.read_table(loc, columns=["image_id","source_real_id","generator","label","sha256"])
        rows += list(zip(t.column("image_id").to_pylist(), t.column("source_real_id").to_pylist(),
                         t.column("generator").to_pylist(), t.column("label").to_pylist(),
                         t.column("sha256").to_pylist()))
        cnt += t.num_rows
    print(f"{p:20} {cnt:6} rows  ({len(shards)} shards)")
print("total:", len(rows))

real/real-70beac4d-00000.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

real/real-70beac4d-00001.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

real/real-70beac4d-00002.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

real/real-70beac4d-00003.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

real/real-70beac4d-00004.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

...

data/sd_cascade/sd_cascade-fcbce73c-0000(…):   0%|          | 0.00/12.3k [00:00<?, ?B/s]

data/sd_cascade/sd_cascade-fcbce73c-0000(…):   0%|          | 0.00/12.2k [00:00<?, ?B/s]

data/sd_cascade/sd_cascade-fcbce73c-0000(…):   0%|          | 0.00/12.1k [00:00<?, ?B/s]

data/sd_cascade/sd_cascade-fcbce73c-0001(…):   0%|          | 0.00/10.4k [00:00<?, ?B/s]

data/sd_cascade/      10000 rows  (196 shards)
total: 70000


In [4]:
# dedup by image_id; flag cross-label byte collisions
seen_id, uniq = set(), []; dup_ids = 0
for r in rows:
    if r[0] in seen_id: dup_ids += 1; continue
    seen_id.add(r[0]); uniq.append(r)
sha_to_ids = defaultdict(set)
for iid,sid,gen,lab,sha in uniq: sha_to_ids[sha].add(iid)
cross = {s:ids for s,ids in sha_to_ids.items() if len(ids)>1}
print(f"unique: {len(uniq)} | dup image_id rows dropped: {dup_ids} | shared-byte groups: {len(cross)}")
for s,ids in list(cross.items())[:3]: print("  shared bytes:", list(ids)[:4])

unique: 70000 | dup image_id rows dropped: 0 | shared-byte groups: 1
  shared bytes: ['sd_cascade_002653', 'sd_cascade_005517', 'sd_cascade_006338', 'sd_cascade_004763']


In [5]:
# deterministic pair-level split keyed on source_real_id + MASTER_SEED
sp   = cfg["split"]
seed = sp.get("seed", MASTER_SEED)
def assign_split(src_id):
    h = int(hashlib.sha256(f"{seed}:{src_id}".encode()).hexdigest()[:8], 16)
    r = (h % 100000) / 100000.0
    return "train" if r < sp["train"] else ("val" if r < sp["train"]+sp["val"] else "test")

split_of = {}
for _,sid,_,_,_ in uniq:
    if sid not in split_of: split_of[sid] = assign_split(sid)
man = [dict(image_id=iid, source_real_id=sid, generator=gen, label=int(lab),
            sha256=sha, split=split_of[sid]) for (iid,sid,gen,lab,sha) in uniq]
from collections import Counter
print("split (images):", Counter(m["split"] for m in man))
print("pairs:", len(split_of), "| per split:", Counter(split_of.values()))
print("per generator:", Counter(m["generator"] for m in man))

split (images): Counter({'train': 49392, 'test': 10486, 'val': 10122})
pairs: 10000 | per split: Counter({'train': 7056, 'test': 1498, 'val': 1446})
per generator: Counter({'real': 10000, 'sd15': 10000, 'sdxl': 10000, 'flux_schnell': 10000, 'kandinsky22': 10000, 'pixart_sigma': 10000, 'sd_cascade': 10000})


In [6]:
import pyarrow as pa, tempfile, os
man_schema = pa.schema([("image_id",pa.string()),("source_real_id",pa.string()),
                        ("generator",pa.string()),("label",pa.int8()),
                        ("sha256",pa.string()),("split",pa.string())])
man_tbl   = pa.Table.from_pylist(man, schema=man_schema)
split_tbl = pa.Table.from_pylist([{"source_real_id":k,"split":v} for k,v in split_of.items()],
                                  schema=pa.schema([("source_real_id",pa.string()),("split",pa.string())]))
api = C.HfApi(token=TOKEN)
for tbl, name in [(man_tbl,"manifest.parquet"),(split_tbl,"splits.parquet")]:
    tmp = os.path.join(tempfile.gettempdir(), name)
    C.pq.write_table(tbl, tmp, compression="zstd")
    api.upload_file(path_or_fileobj=tmp, path_in_repo=name, repo_id=REPO_ID,
                    repo_type="dataset", commit_message=f"write {name}")
    print("uploaded", name, tbl.num_rows, "rows")
print("\nNB09 complete. Next: NB10 (validation + leak audit).")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

uploaded manifest.parquet 70000 rows


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

uploaded splits.parquet 10000 rows

NB09 complete. Next: NB10 (validation + leak audit).
